## dbt Pipeline — Health Data Platform

**What this notebook does:**
```
PostgreSQL (public schema)          PostgreSQL (analytics schema)
───────────────────────────         ──────────────────────────────
customers              ──dbt──►     stg_patients
blood_reports          ──dbt──►     stg_blood_reports        ──►  mart_patient_health
customer_files         ──dbt──►     stg_customer_files       ──►  mart_blood_trends
                                    int_patient_blood_summary
```

**dbt layers:**
```
staging      → clean + rename columns  (views)
intermediate → joins + business logic  (views)
mart         → final analytics tables  (physical tables)
```

**Run order:**
```
Cell 1  → Install + setup
Cell 2  → Create folder structure + all SQL model files
Cell 3  → Test connection (dbt debug)
Cell 4  → Run all models (dbt run)
Cell 5  → Run only specific layer
Cell 6  → Test models (dbt test)
Cell 7  → Generate + serve docs
Cell 8  → Incremental run — use when new data arrives
Cell 9  → Query results in Python
```

## 1. Install & Setup

In [2]:
# !pip install dbt-postgres psycopg2-binary python-dotenv pandas --quiet

In [3]:
# !pip install dbt-postgres

In [4]:
import os
import subprocess
import psycopg2
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
def get_pg():
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

def run(cmd):
    """Run a shell command from the dbt/ folder and print output."""
    result = subprocess.run(
        cmd, shell=True, cwd="dbt",
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

print("✅ Ready")

✅ Ready


---
## 2. Create dbt Project Files
Run **once**. Creates all folders, config files, and SQL model files.

In [11]:
# ── folder structure ─────────────────────────────────────────────────
FOLDERS = [
    "dbt/models/staging",
    "dbt/models/intermediate",
    "dbt/models/mart",
    "dbt/tests",
]
for f in FOLDERS:
    Path(f).mkdir(parents=True, exist_ok=True)
print("✅ Folders created")


# ── dbt_project.yml ──────────────────────────────────────────────────
Path("dbt/dbt_project.yml").write_text("""
name: health_platform
version: '1.0.0'
config-version: 2
profile: health_platform
model-paths: ["models"]
target-path: "target"
clean-targets: ["target", "dbt_packages"]

models:
  health_platform:
    staging:
      +materialized: view
    intermediate:
      +materialized: view
    mart:
      +materialized: table
""", encoding="utf-8")
print("✅ dbt_project.yml created")


# ── profiles.yml ─────────────────────────────────────────────────────
Path("dbt/profiles.yml").write_text(f"""
health_platform:
  target: dev
  outputs:
    dev:
      type: postgres
      host: localhost
      port: 5432
      dbname: {os.getenv('POSTGRES_DB')}
      user: {os.getenv('POSTGRES_USER')}
      password: {os.getenv('POSTGRES_PASSWORD')}
      schema: analytics
      threads: 4
""", encoding="utf-8")
print("✅ profiles.yml created")

✅ Folders created
✅ dbt_project.yml created
✅ profiles.yml created


In [8]:
# ── staging models ────────────────────────────────────────────────────

Path("dbt/models/staging/stg_patients.sql").write_text("""
-- Clean and standardise raw customers table
SELECT
    customer_id,
    name,
    age,
    gender,
    blood_type,
    city,
    first_seen   AS registered_date
FROM public.customers
WHERE customer_id IS NOT NULL
""")

Path("dbt/models/staging/stg_blood_reports.sql").write_text("""
-- Clean blood reports — filter nulls, cast types
SELECT
    id                    AS report_id,
    customer_id,
    report_date::DATE     AS report_date,
    lab_name,
    glucose,
    hemoglobin,
    cholesterol_total,
    cholesterol_hdl,
    cholesterol_ldl,
    triglycerides,
    wbc,
    rbc,
    is_valid,
    minio_path,
    indexed_at
FROM public.blood_reports
WHERE customer_id  IS NOT NULL
  AND report_date  IS NOT NULL
""")

Path("dbt/models/staging/stg_customer_files.sql").write_text("""
-- File metadata from MinIO
SELECT
    customer_id,
    file_name,
    data_type,
    object_path,
    size_bytes,
    file_ext,
    last_modified,
    indexed_at
FROM public.customer_files
""")

print("✅ Staging models created")


# ── intermediate model ────────────────────────────────────────────────

Path("dbt/models/intermediate/int_patient_blood_summary.sql").write_text("""
-- Join patients with blood report averages + abnormal flags
SELECT
    p.customer_id,
    p.name,
    p.age,
    p.gender,
    p.blood_type,
    p.city,
    COUNT(b.report_id)                           AS total_reports,
    ROUND(AVG(b.glucose)::numeric,        2)     AS avg_glucose,
    ROUND(AVG(b.hemoglobin)::numeric,     2)     AS avg_hemoglobin,
    ROUND(AVG(b.cholesterol_total)::numeric, 2)  AS avg_cholesterol,
    MAX(b.report_date)                           AS latest_report_date,
    SUM(CASE WHEN b.glucose          > 126 THEN 1 ELSE 0 END) AS high_glucose_count,
    SUM(CASE WHEN b.cholesterol_total > 200 THEN 1 ELSE 0 END) AS high_cholesterol_count,
    SUM(CASE WHEN b.hemoglobin        < 12  THEN 1 ELSE 0 END) AS low_hemoglobin_count
FROM {{ ref('stg_patients') }} p
LEFT JOIN {{ ref('stg_blood_reports') }} b
    ON p.customer_id = b.customer_id
GROUP BY
    p.customer_id, p.name, p.age,
    p.gender, p.blood_type, p.city
""")

print("✅ Intermediate model created")


# ── mart models ───────────────────────────────────────────────────────

Path("dbt/models/mart/mart_patient_health.sql").write_text("""
-- Final patient health summary — one row per patient
SELECT
    customer_id,
    name,
    age,
    gender,
    blood_type,
    city,
    total_reports,
    avg_glucose,
    avg_hemoglobin,
    avg_cholesterol,
    latest_report_date,
    high_glucose_count,
    high_cholesterol_count,
    low_hemoglobin_count,
    CASE
        WHEN high_glucose_count      > 0
          OR high_cholesterol_count  > 0
          OR low_hemoglobin_count    > 0
        THEN 'AT RISK'
        ELSE 'NORMAL'
    END                 AS health_status,
    NOW()               AS last_updated
FROM {{ ref('int_patient_blood_summary') }}
""")

Path("dbt/models/mart/mart_blood_trends.sql").write_text("""
-- Monthly blood test trends per patient
SELECT
    customer_id,
    DATE_TRUNC('month', report_date)             AS month,
    COUNT(*)                                     AS reports_count,
    ROUND(AVG(glucose)::numeric,          2)     AS avg_glucose,
    ROUND(AVG(hemoglobin)::numeric,       2)     AS avg_hemoglobin,
    ROUND(AVG(cholesterol_total)::numeric, 2)    AS avg_cholesterol,
    ROUND(AVG(triglycerides)::numeric,    2)     AS avg_triglycerides,
    SUM(CASE WHEN glucose > 126           THEN 1 ELSE 0 END) AS abnormal_glucose,
    SUM(CASE WHEN cholesterol_total > 200 THEN 1 ELSE 0 END) AS abnormal_cholesterol
FROM {{ ref('stg_blood_reports') }}
GROUP BY customer_id, DATE_TRUNC('month', report_date)
ORDER BY customer_id, month
""")

print("✅ Mart models created")
print("\n📁 dbt project structure ready:")
for f in sorted(Path("dbt").rglob("*")):
    print(f"  {f}")

✅ Staging models created
✅ Intermediate model created
✅ Mart models created

📁 dbt project structure ready:
  dbt\dbt_project.yml
  dbt\models
  dbt\models\intermediate
  dbt\models\intermediate\int_patient_blood_summary.sql
  dbt\models\mart
  dbt\models\mart\mart_blood_trends.sql
  dbt\models\mart\mart_patient_health.sql
  dbt\models\staging
  dbt\models\staging\stg_blood_reports.sql
  dbt\models\staging\stg_customer_files.sql
  dbt\models\staging\stg_patients.sql
  dbt\profiles.yml
  dbt\tests


---
## 3. Test Connection — `dbt debug`
Confirms dbt can reach PostgreSQL before running anything.

In [9]:
run("dbt debug")

02:58:49  Running with dbt=1.12.0-b1
02:58:49  dbt version: 1.12.0-b1
02:58:49  python version: 3.12.0
02:58:49  python path: C:\Users\Nirasha J\Personal HealthData Platform\.venv\Scripts\python.exe
02:58:49  os info: Windows-11-10.0.26200-SP0
02:58:50  Using profiles dir at c:\Users\Nirasha J\Personal HealthData Platform\dbt
02:58:50  Using profiles.yml file at c:\Users\Nirasha J\Personal HealthData Platform\dbt\profiles.yml
02:58:50  Using dbt_project.yml file at c:\Users\Nirasha J\Personal HealthData Platform\dbt\dbt_project.yml
02:58:50  adapter type: postgres
02:58:50  adapter version: 1.10.0
02:58:50  Configuration:
02:58:50    profiles.yml file [OK found and valid]
02:58:50    dbt_project.yml file [OK found and valid]
02:58:50  Required dependencies:
02:58:50   - git [OK found]

02:58:50  Connection:
02:58:50    host: localhost
02:58:50    port: 5432
02:58:50    user: admin
02:58:50    database: healthcare
02:58:50    schema: analytics
02:58:50    connect_timeout: 10
02:58:50   

0

---
## 4. Run All Models — `dbt run`
Runs all 6 models in dependency order automatically.

In [10]:
run("dbt run")

03:13:33  Running with dbt=1.12.0-b1
03:13:34  Registered adapter: postgres=1.10.0
03:13:34  Unable to do partial parsing because saved manifest not found. Starting full parse.
03:13:34  [ERROR]: Encountered an error:
'utf-8' codec can't decode byte 0x97 in position 34: invalid start byte
03:13:34  Traceback (most recent call last):
  File "C:\Users\Nirasha J\Personal HealthData Platform\.venv\Lib\site-packages\dbt\cli\requires.py", line 184, in wrapper
    result, success = func(*args, **kwargs)
                      ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nirasha J\Personal HealthData Platform\.venv\Lib\site-packages\dbt\cli\requires.py", line 130, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nirasha J\Personal HealthData Platform\.venv\Lib\site-packages\dbt\cli\requires.py", line 281, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nirasha J\Personal HealthData Platform\.venv\Lib\site-packa

2

---
## 5. Run Specific Layer Only

In [ ]:
# ✏️ change 'staging' to 'intermediate' or 'mart' as needed
run("dbt run --select staging")

In [ ]:
run("dbt run --select mart")

---
## 6. Add & Run Tests
dbt tests verify data quality — not null, unique, accepted values.

In [ ]:
# create schema.yml with built-in dbt tests
Path("dbt/models/staging/schema.yml").write_text("""
version: 2

models:
  - name: stg_patients
    columns:
      - name: customer_id
        tests:
          - not_null
          - unique

  - name: stg_blood_reports
    columns:
      - name: report_id
        tests:
          - not_null
          - unique
      - name: customer_id
        tests:
          - not_null
      - name: report_date
        tests:
          - not_null
""")

Path("dbt/models/mart/schema.yml").write_text("""
version: 2

models:
  - name: mart_patient_health
    columns:
      - name: customer_id
        tests:
          - not_null
          - unique
      - name: health_status
        tests:
          - accepted_values:
              values: ['AT RISK', 'NORMAL']
""")

print("✅ schema.yml test files created")
run("dbt test")

---
## 7. Generate Documentation & Lineage Graph
Opens a visual lineage graph at http://localhost:8080

In [ ]:
run("dbt docs generate")
print("\n📖 Docs generated.")
print("   Run this in a separate terminal to view:")
print("   cd dbt && dbt docs serve --port 8080")
print("   Then open: http://localhost:8080")

---
## 8. ▶️ Incremental Run — Use When New Data Arrives

Run this cell whenever new patients or blood reports are added.
- Staging + intermediate views update automatically (they're views — always fresh)
- Mart tables rebuild only changed rows

In [ ]:
def incremental_run():
    """Re-run dbt models to pick up any new data."""
    print("🔄 Running incremental dbt update...\n")
    code = run("dbt run")
    if code == 0:
        print("✅ All models updated successfully")
    else:
        print("❌ Some models failed — check output above")


incremental_run()

---
## 9. Query Results in Python
Verify what dbt built in the analytics schema.

In [ ]:
def query(sql, label):
    pg = get_pg()
    df = pd.read_sql(sql, pg)
    pg.close()
    print(f"\n📊 {label}")
    print(df.to_string(index=False))


# patient health summary
query("""
    SELECT customer_id, name, age,
           avg_glucose, avg_hemoglobin,
           health_status, latest_report_date
    FROM analytics.mart_patient_health
    ORDER BY health_status DESC
""", "Patient Health Summary")


# patients AT RISK
query("""
    SELECT customer_id, name,
           high_glucose_count,
           high_cholesterol_count,
           low_hemoglobin_count
    FROM analytics.mart_patient_health
    WHERE health_status = 'AT RISK'
""", "At Risk Patients")


# monthly blood trends
query("""
    SELECT customer_id, month,
           avg_glucose, avg_cholesterol,
           abnormal_glucose, abnormal_cholesterol
    FROM analytics.mart_blood_trends
    ORDER BY customer_id, month DESC
""", "Monthly Blood Trends")